# Build a PDF Knowledge Base

Most real-world RAG applications start with PDF files — product manuals, research papers, legal contracts, financial reports. This notebook walks through ingesting a PDF, splitting it intelligently, storing vectors in a persistent Chroma database, and querying the result.

**What you'll build:** A knowledge base from a PDF that survives process restarts and answers natural-language questions.

**Time:** ~15 min | **Difficulty:** Beginner

Guide: [Build a PDF Knowledge Base](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/pdf-knowledge-base)

## Prerequisites & Setup

You will need:
- An **OpenAI API key** — set `OPENAI_API_KEY` in the env cell below
- A PDF file to ingest — update `PDF_PATH` to point to your file
- `synapsekit`, `chromadb`, and `pypdf` installed (run the install cell)

**What you'll learn:**
- How to load a PDF with `PDFLoader`
- How `RecursiveCharacterTextSplitter` preserves paragraph boundaries
- How to persist vectors with `ChromaVectorStore` so they survive restarts
- How to attach metadata (page number, source filename) to every chunk
- How to query with source attribution

In [ ]:
!pip install synapsekit chromadb pypdf -q

In [ ]:
import os

# Set your OpenAI API key before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# Path to the PDF file you want to ingest
PDF_PATH = "company-handbook.pdf"
PERSIST_DIR = "./chroma_db"

## Step 1: Load the PDF

`PDFLoader` returns a list of `Document` objects — one per page. Each `Document` has `.page_content` (the text) and `.metadata` (a dict you can query against later).

In [ ]:
from synapsekit.loaders import PDFLoader

# PDFLoader preserves page numbers as metadata so you can cite sources later.
# Async loading avoids blocking the event loop on large files.
loader = PDFLoader(PDF_PATH)
documents = await loader.aload()

print(f"Loaded {len(documents)} pages")
print(documents[0].metadata)
# {'source': 'company-handbook.pdf', 'page': 1}

## Step 2: Split Pages into Chunks

`RecursiveCharacterTextSplitter` tries to split on paragraph breaks first, then sentences, then words — preserving as much semantic coherence as possible. Raw character splitting would break sentences mid-word, degrading retrieval quality.

In [ ]:
from synapsekit.splitters import RecursiveCharacterTextSplitter

# chunk_size=1000 fits comfortably in the context window while staying large
# enough to preserve complete sentences and paragraph structure.
# chunk_overlap=200 ensures a sentence cut at a boundary still appears in full
# in at least one chunk, preventing information loss.
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(chunks[0].page_content[:200])

## Step 3: Set Up a Persistent Vector Store

Persisting to disk is the single most important change from the quickstart. Without persistence you re-embed every time the process restarts, which costs money and adds startup latency.

In [ ]:
from synapsekit.vectorstores.chroma import ChromaVectorStore
from synapsekit.embeddings.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# persist_directory keeps the vectors on disk so you only pay for embedding once.
# Subsequent restarts load from disk in milliseconds rather than re-embedding.
vectorstore = ChromaVectorStore(
    collection_name="company-handbook",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

## Step 4: Embed and Ingest Chunks

On first run this call sends your chunks to the OpenAI embeddings API and writes the resulting vectors to `./chroma_db`. On subsequent runs you skip this step entirely and load directly from disk.

In [ ]:
# Batched ingestion respects the OpenAI embedding API rate limit automatically.
await vectorstore.aadd_documents(chunks)
print("Ingestion complete.")

## Step 5: Build the RAG Pipeline

The pipeline reuses the same `vectorstore` instance, so no data is copied. Swapping the LLM later (e.g., to `AnthropicLLM`) requires changing only one line here.

In [ ]:
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=embeddings,
    vectorstore=vectorstore,
)

## Step 6: Query with Source Attribution

`return_sources=True` makes the pipeline return `(answer, sources)` instead of just the answer string, so you can show users where the answer came from.

In [ ]:
# return_sources=True makes the pipeline return (answer, sources) instead of
# just the answer string, so you can show users where the answer came from.
answer, sources = await rag.aquery(
    "What is the company's remote work policy?",
    return_sources=True,
)

print("Answer:", answer)
print("\nSources:")
for doc in sources:
    print(f"  - {doc.metadata['source']}, page {doc.metadata['page']}")

## Step 7: Re-Use an Existing Chroma Database

On subsequent runs, skip `aadd_documents()` and just point at the existing db. Chroma loads the index from disk without hitting the embeddings API.

In [ ]:
# On subsequent runs, skip aadd_documents() and just point at the existing db.
# Chroma loads the index from disk without hitting the embeddings API.
existing_vectorstore = ChromaVectorStore(
    collection_name="company-handbook",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=embeddings,
    vectorstore=existing_vectorstore,
)

answer = await rag.aquery("How many days of paid leave do employees get?")
print(answer)

## Complete Working Example

A single self-contained cell that runs the entire pipeline end-to-end. Update `PDF_PATH` to point to your PDF file before running.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.loaders import PDFLoader
from synapsekit.splitters import RecursiveCharacterTextSplitter
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.llms.openai import OpenAILLM
from synapsekit.vectorstores.chroma import ChromaVectorStore

PERSIST_DIR = "./chroma_db"
PDF_PATH = "company-handbook.pdf"  # <-- update this path

async def ingest(vectorstore):
    loader = PDFLoader(PDF_PATH)
    documents = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    await vectorstore.aadd_documents(chunks)
    print(f"Ingested {len(chunks)} chunks from {len(documents)} pages.")

async def main():
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = ChromaVectorStore(
        collection_name="company-handbook",
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )

    await ingest(vectorstore)

    rag = RAGPipeline(
        llm=OpenAILLM(model="gpt-4o-mini"),
        embeddings=embeddings,
        vectorstore=vectorstore,
    )

    answer, sources = await rag.aquery(
        "What is the company's remote work policy?",
        return_sources=True,
    )
    print("Answer:", answer)
    for doc in sources:
        print(f"  Source: {doc.metadata['source']}, page {doc.metadata['page']}")

asyncio.run(main())

## Next Steps

- [Multi-Format Document Ingestion](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/multi-format-ingestion) — add DOCX, web pages, and CSVs alongside PDFs
- [Choosing a Chunking Strategy](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/chunking-strategies) — understand when to use a different splitter
- [Metadata Filtering in Vector Search](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/metadata-filtering) — scope queries to specific pages or sections